In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

result = search_tool.invoke("Vijay rally in karur")
print(result)

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = "your-api-key"

llm = init_chat_model("openai:gpt-4.1")

# Tool Use

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition


@tool
def check_ticket_status(ticket_id: str) -> str:
    """
    Ticket category is FQASL.
    Returns the status of a support ticket as a string.
    """
    ticket_status = {
        "SUP-20251008-7A4F2C": "Open",
        "BILL-20251007-3B9E1D": "In Progress",
        "TECH-20251005-9C8F7A": "Resolved",
        "SUP-20251006-2D4C9B": "Closed"
    }
    
    return ticket_status.get(ticket_id, "Ticket ID not found")
    
# Augment the LLM with tools
tools = [check_ticket_status,search_tool]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
tools_by_name = {tool.name: tool for tool in tools}

tools_by_name

In [ ]:
tool = tools_by_name["check_ticket_status"]
observation = tool.invoke("SUP-20251006-2D4C9B")
observation

In [ ]:
# import langchain
# langchain.debug = True

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.memory import MemorySaver
from typing import Literal
from langchain_core.messages import ToolMessage

class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)


customerSupport_template = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer support assistant for DummyCorp. Give clear, polite, and helpful replies in maximum 20 words."),
    ("human", "{input}")
])

customerSupport_chain = customerSupport_template | llm_with_tools

# Agent
def customerSupportAgent(state: State):
    # print(f"Before State : {state} \n\n")
    response = customerSupport_chain.invoke(state["messages"])
    # print(f"After State : {state} \n\n")
    return {"messages": [response]}


def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}


def should_continue(state: State) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]
    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tool_node"
    # Otherwise, we stop (reply to the user)
    return END

    
graph_builder.add_node("customerSupport", customerSupportAgent)
graph_builder.add_node("tool_node", tool_node)

graph_builder.add_edge(START, "customerSupport")
graph_builder.add_conditional_edges(
    "customerSupport",
    should_continue,
    ["tool_node", END]
)
graph_builder.add_edge("tool_node", "customerSupport")

graph_builder.add_edge("customerSupport", END)

# Compile with memory
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

# Config with thread ID for chat history
config = {"configurable": {"thread_id": "my_conversation"}}


In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
def stream_graph_updates(user_input: str):
    result = graph.invoke(
        {"messages": [{"role": "user", "content": user_input}]},config=config)
    if result["messages"]:
        print("Assistant:", result["messages"][-1].content)

while True:
    try:
        user_input = input("\nUser: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except:
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break

In [ ]:
def stream_graph_updates(user_input: str):
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}, config=config):
        print("Event: ",event,"\n\n")
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)


while True:
    try:
        user_input = input("User: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except:
        # fallback if input() is not available
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break